# LeetCode 1717 — Maximum Score From Removing Substrings bằng Python

Link đề: https://leetcode.com/problems/maximum-score-from-removing-substrings/ · Độ khó: **Medium** · Chủ đề: String, Stack, Greedy

Skeleton LeetCode:

```python
class Solution:
    def maximumGain(self, s: str, x: int, y: int) -> int:
        ...
```

Lộ trình: **vét cạn mọi thứ tự xóa** (có ghi nhớ) để có đáp án chuẩn → tham lam bằng `str.replace` lặp lại (`O(n²)`) → **tham lam + ngăn xếp** hai lượt (`O(n)`) → **đếm thuần** không cần ngăn xếp (`O(1)` bộ nhớ). Điểm mấu chốt: *xóa cặp đắt tiền trước* — và ta sẽ chứng minh vì sao điều đó luôn tối ưu.

## 1. Mục tiêu học

1. Hiểu thao tác "xóa chuỗi con rồi nối lại" và vì sao việc xóa có thể **tạo ra** cặp mới (ví dụ `"aabb"` → `"ab"`).
2. Dùng **ngăn xếp** để mô phỏng "xóa liên tục mọi lần xuất hiện của một mẫu 2 ký tự" trong `O(n)` — cùng kỹ thuật với bài khớp ngoặc.
3. Lập luận **tham lam**: ưu tiên mẫu có điểm cao hơn, rồi xử lý mẫu còn lại; chứng minh bằng nhận xét "tổng số lần xóa trong mỗi khúc là cố định = `min(#a, #b)`".
4. Rút gọn ngăn xếp thành **hai biến đếm** vì ngăn xếp chỉ chứa `a` hoặc `b`, và biết **reset** khi gặp ký tự khác `a`/`b`.
5. Kiểm thử tham lam bằng **vét cạn có ghi nhớ** (`lru_cache`) trên chuỗi ngắn ngẫu nhiên.

## 2. Đề bài & ví dụ

Cho chuỗi `s` và hai số nguyên `x`, `y`. Có thể thực hiện hai loại thao tác **bao nhiêu lần tùy ý**:
- Xóa một chuỗi con `"ab"` và được `x` điểm. Ví dụ xóa `"ab"` trong `"c`**`ab`**`xbae"` được `"cxbae"`.
- Xóa một chuỗi con `"ba"` và được `y` điểm. Ví dụ xóa `"ba"` trong `"cabx`**`ba`**`e"` được `"cabxe"`.

Trả về **số điểm lớn nhất** có thể đạt.

**Ví dụ 1:** `s = "cdbcbbaaabab", x = 4, y = 5` → `19`. Một chuỗi thao tác: xóa `"ba"` (+5) → `"cdbcbbaaab"`; xóa `"ab"` (+4) → `"cdbcbbaa"`; xóa `"ba"` (+5) → `"cdbcba"`; xóa `"ba"` (+5) → `"cdbc"`. Tổng `19`.

**Ví dụ 2:** `s = "aabbaaxybbaabb", x = 5, y = 4` → `20`.

**Ràng buộc:**
- `1 ≤ s.length ≤ 10⁵`
- `1 ≤ x, y ≤ 10⁴`
- `s` gồm chữ cái tiếng Anh thường.

In [1]:
import random
import time
from functools import lru_cache
from typing import List

examples = [
    (("cdbcbbaaabab", 4, 5), 19),
    (("aabbaaxybbaabb", 5, 4), 20),
]
for (s, x, y), want in examples:
    print(f"s={s!r}, x={x}, y={y} -> kỳ vọng {want}")

s='cdbcbbaaabab', x=4, y=5 -> kỳ vọng 19
s='aabbaaxybbaabb', x=5, y=4 -> kỳ vọng 20


## 3. Kiến thức nền

### 3.1 Chuỗi bất biến, lát cắt và nối chuỗi

Chuỗi Python **bất biến**: "xóa" `s[k:k+2]` thực chất là tạo chuỗi mới `s[:k] + s[k+2:]` — tốn `O(n)`. Làm điều đó lặp lại nhiều lần sẽ thành `O(n²)`.

In [2]:
s = "cabxbae"
k = s.find("ab")
print("vị trí 'ab':", k, "->", s[:k] + s[k + 2:])
t = "aabb"
k = t.find("ab")
t2 = t[:k] + t[k + 2:]
print(t, "-> xóa 'ab' ->", t2, "-> lại có 'ab' mới:", "ab" in t2)

vị trí 'ab': 1 -> cxbae
aabb -> xóa 'ab' -> ab -> lại có 'ab' mới: True


### 3.2 `str.replace` chỉ thay các lần xuất hiện **không chồng lấn**, một lượt

`"aabb".replace("ab", "")` chỉ xóa `"ab"` ở giữa và trả về `"ab"` — cặp mới sinh ra **không** bị xóa trong cùng lượt. Muốn xóa hết phải lặp đến khi không đổi.

In [3]:
print(repr("aabb".replace("ab", "")))
t = "aaabbb"
rounds = 0
while "ab" in t:
    t = t.replace("ab", "")
    rounds += 1
print("sau", rounds, "lượt:", repr(t))

'ab'
sau 3 lượt: ''


### 3.3 Ngăn xếp bằng `list`

`append` (đẩy) và `pop` (lấy) ở cuối list là `O(1)`. Phần tử trên cùng là `st[-1]`. Ngăn xếp cho phép "nhìn lại ký tự sát bên trái chưa bị xóa" — đúng thứ cần để phát hiện cặp mới hình thành sau khi xóa.

In [4]:
st = []
for c in "aabb":
    if st and st[-1] == "a" and c == "b":
        st.pop()
        print(f"gặp {c!r}: khớp với 'a' trên đỉnh -> xóa 'ab', stack={st}")
    else:
        st.append(c)
        print(f"gặp {c!r}: đẩy vào, stack={st}")

gặp 'a': đẩy vào, stack=['a']
gặp 'a': đẩy vào, stack=['a', 'a']
gặp 'b': khớp với 'a' trên đỉnh -> xóa 'ab', stack=['a']
gặp 'b': khớp với 'a' trên đỉnh -> xóa 'ab', stack=[]


### 3.4 `functools.lru_cache` cho đệ quy có ghi nhớ

Gắn `@lru_cache(maxsize=None)` lên hàm đệ quy có tham số **bất biến** (chuỗi, số…) để mỗi trạng thái chỉ tính một lần. Ta dùng nó cho lời giải vét cạn.

In [5]:
@lru_cache(maxsize=None)
def fib(k):
    return k if k < 2 else fib(k - 1) + fib(k - 2)

print(fib(80), fib.cache_info())

23416728348467685 CacheInfo(hits=78, misses=81, maxsize=None, currsize=81)


### 3.5 Hoán đổi để luôn xử lý "mẫu đắt" trước

Kỹ thuật chuẩn hóa: nếu `x < y`, đổi vai `a ↔ b` và `x ↔ y`. Khi đó mẫu đắt luôn là `first + second` với điểm `x ≥ y`. Python hoán đổi gọn bằng `x, y = y, x`.

In [6]:
x, y = 4, 5
first, second = "a", "b"
if x < y:
    x, y = y, x
    first, second = second, first
print(f"mẫu đắt = {first + second!r} ({x} điểm), mẫu rẻ = {second + first!r} ({y} điểm)")

mẫu đắt = 'ba' (5 điểm), mẫu rẻ = 'ab' (4 điểm)


## 4. Ý tưởng nền tảng / trực giác

**Quan sát 1 — ký tự khác `a`, `b` là bức tường.** Ký tự như `c`, `x` không bao giờ bị xóa, nên một cặp `ab`/`ba` không thể "vượt qua" nó. Chuỗi được chia thành các **khúc** chỉ gồm `a`, `b`, xử lý độc lập.

**Quan sát 2 — mỗi lần xóa lấy đi đúng một `a` và một `b`.** Trong một khúc có `A` chữ `a` và `B` chữ `b`, số lần xóa tối đa là `min(A, B)`. Và thực tế luôn đạt được: chừng nào khúc còn cả `a` lẫn `b`, chắc chắn có chỗ `a` đứng cạnh `b` (hoặc `ab` hoặc `ba`).

⇒ **Tổng số lần xóa cố định**; câu hỏi chỉ là: **bao nhiêu lần trong đó là mẫu đắt?** Càng nhiều mẫu đắt càng tốt.

**Tính tay Ví dụ 1** `"cdbcbbaaabab"`, `x = 4` (`ab`), `y = 5` (`ba`) ⇒ `ba` đắt hơn.
- Khúc chỉ gồm `a/b`: `"b"` (giữa `d` và `c`) và `"bbaaabab"`.
- Khúc `"bbaaabab"` có `A = 4`, `B = 4` ⇒ đúng 4 lần xóa.
- Ưu tiên `ba`: xóa hết `ba` có thể: `bbaaabab` → xóa `ba` ở vị trí 1–2 → `baabab` → xóa `ba` ở đầu → `abab` → xóa `ba` ở giữa → `ab`. Được 3 lần `ba`, không còn `ba` nào.
- Phần còn lại `"ab"` cho 1 lần `ab`. Tổng `3·5 + 1·4 = 19` ✔ (4 lần xóa = `min(A, B)`).

In [7]:
import re
s = "cdbcbbaaabab"
blocks = [blk for blk in re.split(r"[^ab]+", s) if blk]
for blk in blocks:
    print(f"khúc {blk!r}: A={blk.count('a')}, B={blk.count('b')}, số lần xóa = {min(blk.count('a'), blk.count('b'))}")

khúc 'b': A=0, B=1, số lần xóa = 0
khúc 'bbaaabab': A=4, B=4, số lần xóa = 4


## 5. Cách 1 — Brute Force / baseline: thử **mọi** thứ tự xóa (có ghi nhớ)

**Ý tưởng:** từ chuỗi hiện tại `t`, thử xóa từng vị trí có `"ab"` hoặc `"ba"`, đệ quy trên chuỗi mới, lấy max. Ghi nhớ theo `t` để không tính lại trạng thái trùng.

Đây là định nghĩa "đúng theo đề" nhất — không giả định gì về chiến lược — nên dùng làm **đáp án chuẩn**.

In [8]:
def gain_brute(s, x, y):
    @lru_cache(maxsize=None)
    def best(t):
        res = 0
        for k in range(len(t) - 1):
            pair = t[k:k + 2]
            if pair == "ab":
                res = max(res, x + best(t[:k] + t[k + 2:]))
            elif pair == "ba":
                res = max(res, y + best(t[:k] + t[k + 2:]))
        return res
    return best(s)

for (s, x, y), want in examples:
    print(gain_brute(s, x, y), "| kỳ vọng", want)

19 | kỳ vọng 19
20 | kỳ vọng 20


**Độ phức tạp:** số trạng thái (chuỗi con có thể đạt được) tăng **theo hàm mũ** trong trường hợp xấu; mỗi trạng thái tốn `O(n²)` (thử `n` vị trí, mỗi lần tạo chuỗi `O(n)`). Chỉ dùng được với `n ≲ 14`.

**Nút thắt:** ta đang thử mọi thứ tự, trong khi Quan sát 2 cho biết tổng số lần xóa là cố định — chỉ cần một **chiến lược** đúng để tối đa số lần xóa mẫu đắt.

In [9]:
for n in [8, 10, 12, 14]:
    s = ("ab" * n)[:n]
    t = time.perf_counter()
    r = gain_brute(s, 3, 2)
    print(f"n={n:>2}: {time.perf_counter() - t:.3f}s -> {r}")

n= 8: 0.000s -> 12
n=10: 0.000s -> 15
n=12: 0.000s -> 18
n=14: 0.000s -> 21


## 6. Ý tưởng thuật toán / bước đột phá

Chuẩn hóa để mẫu đắt là `P = first+second` (điểm `x`), mẫu rẻ là `Q = second+first` (điểm `y`), `x ≥ y`.

**Insight (tham lam):** Trong một khúc chỉ gồm `first`/`second`:
1. **Xóa `P` nhiều nhất có thể trước.** Coi `first` như `(` và `second` như `)`. Mỗi lần xóa `P` ghép một `(` với một `)` **đứng sau nó**. Số cặp ghép tối đa chính là số cặp ngoặc khớp khi quét trái → phải bằng ngăn xếp — ký hiệu `k`.
2. Sau khi xóa hết `P`, phần còn lại có dạng `second…second first…first` (mọi `)` chưa khớp đứng trước mọi `(` chưa khớp). Mỗi cặp `second` + `first` kề nhau là `Q` ⇒ xóa được `min(#second_còn, #first_còn)` lần — đúng bằng `min(A, B) − k`.
3. Điểm = `x·k + y·(min(A, B) − k)`. Vì `x ≥ y` và `min(A, B)` cố định, điểm tăng theo `k` ⇒ `k` lớn nhất là tối ưu.

> **Quy tắc:** *Nếu* `x ≥ y` *thì* xóa hết `"ab"` trước rồi xóa `"ba"`; *ngược lại* xóa hết `"ba"` trước rồi `"ab"`.

**Invariant của ngăn xếp (lượt 1):** sau khi đọc tiền tố `s[0..i]`, ngăn xếp là chuỗi thu được khi xóa `P` liên tục **hết mức** trong tiền tố đó, và nó **không chứa `P`** nào.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Tham lam bằng `replace` lặp lại

**Từng đoạn:**
1. Chuẩn hóa mẫu đắt `P`, mẫu rẻ `Q`.
2. Lặp: đếm số lần `P` xuất hiện (không chồng lấn) bằng `t.count(P)`, cộng điểm, `t = t.replace(P, "")`; dừng khi không còn `P`.
3. Làm tương tự với `Q` trên chuỗi còn lại.

Đúng về mặt chiến lược nhưng mỗi lượt `O(n)` và có thể cần `~n/2` lượt (ví dụ `"aaa…abbb…b"`) ⇒ `O(n²)`.

In [10]:
def gain_replace(s, x, y):
    P, Q = "ab", "ba"
    if x < y:
        P, Q, x, y = Q, P, y, x
    total = 0
    for pat, pts in ((P, x), (Q, y)):
        while True:
            c = s.count(pat)
            if c == 0:
                break
            total += c * pts
            s = s.replace(pat, "")
    return total

for (s, x, y), want in examples:
    print(gain_replace(s, x, y), "| kỳ vọng", want)

for half in [2000, 4000, 8000]:
    s = "a" * half + "b" * half
    t = time.perf_counter()
    gain_replace(s, 2, 1)
    print(f"n={2 * half:>6}: {time.perf_counter() - t:.3f}s (tăng ~4 lần khi n gấp đôi)")

19 | kỳ vọng 19
20 | kỳ vọng 20
n=  4000: 0.014s (tăng ~4 lần khi n gấp đôi)
n=  8000: 0.055s (tăng ~4 lần khi n gấp đôi)


n= 16000: 0.217s (tăng ~4 lần khi n gấp đôi)


**Độ phức tạp Cách 2:** Time `O(n²)` worst case, Space `O(n)`. Với `n = 10⁵` dạng xấu thì quá chậm.

### 7.2 Cách 3 — Tham lam + ngăn xếp, hai lượt

**Từng đoạn:**
1. Hàm `remove_all(t, p, q)` quét `t`; nếu đỉnh ngăn xếp là `p` và ký tự hiện tại là `q` thì `pop` (xóa cặp `pq`) và đếm +1; ngược lại `append`. Trả về `(chuỗi còn lại, số lần xóa)`.
2. Lượt 1: `remove_all(s, first, second)` với mẫu đắt.
3. Lượt 2: `remove_all(phần còn lại, second, first)` với mẫu rẻ.
4. Điểm = `x·c1 + y·c2`.

Ngăn xếp tự động xử lý cặp mới sinh ra: sau khi `pop`, đỉnh mới là ký tự sát bên trái "vừa lộ ra", sẵn sàng ghép với ký tự kế tiếp. Ký tự khác `a`/`b` được đẩy vào và chặn mọi ghép cặp vượt qua nó — đúng Quan sát 1.

In [11]:
def remove_all(t, p, q):
    st = []
    cnt = 0
    for c in t:
        if c == q and st and st[-1] == p:
            st.pop()
            cnt += 1
        else:
            st.append(c)
    return "".join(st), cnt

def gain_stack(s, x, y):
    first, second = "a", "b"
    if x < y:
        first, second, x, y = second, first, y, x
    rest, c1 = remove_all(s, first, second)
    _, c2 = remove_all(rest, second, first)
    return c1 * x + c2 * y

print(remove_all("cdbcbbaaabab", "b", "a"))
for (s, x, y), want in examples:
    print(gain_stack(s, x, y), "| kỳ vọng", want)

('cdbcab', 3)
19 | kỳ vọng 19
20 | kỳ vọng 20


Code nộp LeetCode của Cách 3:

In [12]:
class SolutionStack:
    def maximumGain(self, s: str, x: int, y: int) -> int:
        first, second = "a", "b"
        if x < y:
            first, second, x, y = second, first, y, x

        def remove_all(t: str, p: str, q: str):
            st = []
            cnt = 0
            for c in t:
                if c == q and st and st[-1] == p:
                    st.pop()
                    cnt += 1
                else:
                    st.append(c)
            return st, cnt

        rest, c1 = remove_all(s, first, second)
        _, c2 = remove_all(rest, second, first)
        return c1 * x + c2 * y

print([SolutionStack().maximumGain(*inp) for inp, _ in examples])

[19, 20]


**Độ phức tạp Cách 3:** Time `O(n)` (mỗi ký tự được đẩy/lấy tối đa một lần mỗi lượt), Space `O(n)` cho ngăn xếp.

### 7.3 Cách 4 — Đếm thuần, không cần ngăn xếp (`O(1)` bộ nhớ)

Trong một khúc, ngăn xếp của lượt 1 chỉ có thể là `second…second first…first`. Vậy chỉ cần hai biến:
- `cf` = số `first` đang chờ trên đỉnh (chưa ghép),
- `cs` = số `second` bị "kẹt" bên dưới (không có `first` nào trước nó để ghép).

**Từng đoạn** (quét từng ký tự `c`):
1. `c == first`: `cf += 1`.
2. `c == second`: nếu `cf > 0` thì ghép mẫu đắt: `ans += x`, `cf -= 1`; ngược lại `cs += 1`.
3. `c` khác: khúc kết thúc ⇒ phần còn lại là `second^cs first^cf` ⇒ cộng `y·min(cf, cs)`, reset `cf = cs = 0`.
4. Cuối chuỗi: cộng `y·min(cf, cs)` cho khúc cuối.

In [13]:
def gain_count(s, x, y):
    first, second = "a", "b"
    if x < y:
        first, second, x, y = second, first, y, x
    ans = cf = cs = 0
    for c in s:
        if c == first:
            cf += 1
        elif c == second:
            if cf:
                ans += x
                cf -= 1
            else:
                cs += 1
        else:
            ans += y * min(cf, cs)
            cf = cs = 0
    ans += y * min(cf, cs)
    return ans

for (s, x, y), want in examples:
    print(gain_count(s, x, y), "| kỳ vọng", want)

19 | kỳ vọng 19
20 | kỳ vọng 20


Code nộp LeetCode của Cách 4 (bản nộp chính, xem lại ở mục 9).

**Độ phức tạp Cách 4:** Time `O(n)`, Space `O(1)`.

## 8. TRACE từng bước

Trace Cách 4 trên Ví dụ 2: `s = "aabbaaxybbaabb"`, `x = 5`, `y = 4` ⇒ `x ≥ y` nên `first = 'a'`, `second = 'b'`, mẫu đắt `"ab"`.

| `i` | `c` | hành động | `cf` | `cs` | `ans` |
|---|---|---|---|---|---|
| 0 | a | cf+1 | 1 | 0 | 0 |
| 1 | a | cf+1 | 2 | 0 | 0 |
| 2 | b | ghép `ab` +5 | 1 | 0 | 5 |
| 3 | b | ghép `ab` +5 | 0 | 0 | 10 |
| 4 | a | cf+1 | 1 | 0 | 10 |
| 5 | a | cf+1 | 2 | 0 | 10 |
| 6 | x | hết khúc: +4·min(2,0) | 0 | 0 | 10 |
| 7 | y | hết khúc | 0 | 0 | 10 |
| 8 | b | không có `a` chờ: cs+1 | 0 | 1 | 10 |
| 9 | b | cs+1 | 0 | 2 | 10 |
| 10 | a | cf+1 | 1 | 2 | 10 |
| 11 | a | cf+1 | 2 | 2 | 10 |
| 12 | b | ghép `ab` +5 | 1 | 2 | 15 |
| 13 | b | ghép `ab` +5 | 0 | 2 | 20 |
| — | hết | +4·min(0,2) = 0 | | | **20** |

In [14]:
def trace_count(s, x, y):
    first, second = "a", "b"
    if x < y:
        first, second, x, y = second, first, y, x
    print(f"mẫu đắt {first + second!r}={x}, mẫu rẻ {second + first!r}={y}")
    ans = cf = cs = 0
    for i, c in enumerate(s):
        if c == first:
            cf += 1; act = "cf+1"
        elif c == second:
            if cf:
                ans += x; cf -= 1; act = f"ghép {first + second} +{x}"
            else:
                cs += 1; act = "cs+1"
        else:
            add = y * min(cf, cs)
            ans += add; act = f"hết khúc: +{y}*min({cf},{cs})={add}"
            cf = cs = 0
        print(f"i={i:>2} c={c} {act:<28} cf={cf} cs={cs} ans={ans}")
    add = y * min(cf, cs)
    ans += add
    print(f"cuối chuỗi: +{y}*min({cf},{cs})={add} -> ans={ans}")
    return ans

trace_count("aabbaaxybbaabb", 5, 4)

mẫu đắt 'ab'=5, mẫu rẻ 'ba'=4
i= 0 c=a cf+1                         cf=1 cs=0 ans=0
i= 1 c=a cf+1                         cf=2 cs=0 ans=0
i= 2 c=b ghép ab +5                   cf=1 cs=0 ans=5
i= 3 c=b ghép ab +5                   cf=0 cs=0 ans=10
i= 4 c=a cf+1                         cf=1 cs=0 ans=10
i= 5 c=a cf+1                         cf=2 cs=0 ans=10
i= 6 c=x hết khúc: +4*min(2,0)=0      cf=0 cs=0 ans=10
i= 7 c=y hết khúc: +4*min(0,0)=0      cf=0 cs=0 ans=10
i= 8 c=b cs+1                         cf=0 cs=1 ans=10
i= 9 c=b cs+1                         cf=0 cs=2 ans=10
i=10 c=a cf+1                         cf=1 cs=2 ans=10
i=11 c=a cf+1                         cf=2 cs=2 ans=10
i=12 c=b ghép ab +5                   cf=1 cs=2 ans=15
i=13 c=b ghép ab +5                   cf=0 cs=2 ans=20
cuối chuỗi: +4*min(0,2)=0 -> ans=20


20

Trace Cách 3 (ngăn xếp) trên Ví dụ 1 để thấy cặp **mới hình thành** sau khi `pop`:

In [15]:
def trace_stack(t, p, q):
    st = []; cnt = 0
    for i, c in enumerate(t):
        if c == q and st and st[-1] == p:
            st.pop(); cnt += 1
            print(f"i={i:>2} c={c}: đỉnh là {p!r} -> xóa {p + q!r}, stack={''.join(st)!r}, cnt={cnt}")
        else:
            st.append(c)
            print(f"i={i:>2} c={c}: đẩy vào, stack={''.join(st)!r}")
    return "".join(st), cnt

rest, c1 = trace_stack("cdbcbbaaabab", "b", "a")      # x=4 < y=5 -> 'ba' đắt
print("lượt 1:", rest, c1)
rest2, c2 = trace_stack(rest, "a", "b")
print("lượt 2:", rest2, c2, "| tổng điểm =", c1 * 5 + c2 * 4)

i= 0 c=c: đẩy vào, stack='c'
i= 1 c=d: đẩy vào, stack='cd'
i= 2 c=b: đẩy vào, stack='cdb'
i= 3 c=c: đẩy vào, stack='cdbc'
i= 4 c=b: đẩy vào, stack='cdbcb'
i= 5 c=b: đẩy vào, stack='cdbcbb'
i= 6 c=a: đỉnh là 'b' -> xóa 'ba', stack='cdbcb', cnt=1
i= 7 c=a: đỉnh là 'b' -> xóa 'ba', stack='cdbc', cnt=2
i= 8 c=a: đẩy vào, stack='cdbca'
i= 9 c=b: đẩy vào, stack='cdbcab'
i=10 c=a: đỉnh là 'b' -> xóa 'ba', stack='cdbca', cnt=3
i=11 c=b: đẩy vào, stack='cdbcab'
lượt 1: cdbcab 3
i= 0 c=c: đẩy vào, stack='c'
i= 1 c=d: đẩy vào, stack='cd'
i= 2 c=b: đẩy vào, stack='cdb'
i= 3 c=c: đẩy vào, stack='cdbc'
i= 4 c=a: đẩy vào, stack='cdbca'
i= 5 c=b: đỉnh là 'a' -> xóa 'ab', stack='cdbc', cnt=1
lượt 2: cdbc 1 | tổng điểm = 19


## 9. Code nộp LeetCode

In [16]:
class Solution:
    def maximumGain(self, s: str, x: int, y: int) -> int:
        first, second = "a", "b"
        if x < y:                                   # để "first+second" luôn là mẫu đắt
            first, second, x, y = second, first, y, x
        ans = cf = cs = 0
        for c in s:
            if c == first:
                cf += 1
            elif c == second:
                if cf:                              # ghép mẫu đắt
                    ans += x
                    cf -= 1
                else:
                    cs += 1
            else:                                   # bức tường: chốt khúc
                ans += y * min(cf, cs)
                cf = cs = 0
        return ans + y * min(cf, cs)

sol = Solution()
for inp, want in examples:
    assert sol.maximumGain(*inp) == want
print("Solution qua 2 ví dụ của đề")

Solution qua 2 ví dụ của đề


## 10. Vì sao thuật toán đúng

Xét một khúc `B` chỉ gồm `first`/`second`, có `A` chữ `first` và `C` chữ `second`. Gọi `P = first+second` (điểm `x`), `Q = second+first` (điểm `y`), `x ≥ y`.

**(1) Mọi chuỗi thao tác xóa đúng `≤ min(A, C)` cặp,** và luôn có thể xóa đúng `min(A, C)`: còn cả hai loại chữ thì phải có một chỗ chuyển `first↔second` kề nhau.

**(2) Số lần xóa `P` trong bất kỳ chuỗi thao tác nào `≤ k`**, với `k` = số cặp khớp khi coi `first` là `(`, `second` là `)`. Lý do: mỗi lần xóa `P` ghép một `first` với một `second` đứng **sau** nó trong chuỗi gốc, các cặp này rời nhau ⇒ tạo thành một ghép cặp "`(` trước `)`"; số cặp như vậy lớn nhất bằng số khớp tham lam của ngăn xếp (bài toán khớp ngoặc kinh điển).

**(3) Thuật toán đạt cả hai cận:** lượt 1 xóa đúng `k` lần `P`; phần còn lại là `second^{C−k} first^{A−k}` (nếu còn `first` đứng trước `second` thì ngăn xếp đã ghép), cho đúng `min(A, C) − k` lần `Q`. Tổng số lần xóa `= min(A, C)`.

**(4) Tối ưu:** mọi chiến lược có `j ≤ k` lần `P` và `≤ min(A,C) − j` lần `Q` nên điểm `≤ x·j + y·(min(A,C) − j) ≤ x·k + y·(min(A,C) − k)` vì `x ≥ y`. Thuật toán đạt đúng vế phải.

Các khúc độc lập (ký tự khác không bao giờ bị xóa nên không cặp nào vượt qua nó), nên cộng điểm từng khúc là tối ưu toàn cục. Cách 4 chỉ là cách đếm `k` và phần còn lại mà không lưu ngăn xếp.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(s)` (`≤ 10⁵`).

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Vét cạn có nhớ | hàm mũ | hàm mũ | chỉ `n ≲ 14` |
| 2. `replace` lặp | `O(n²)` worst | `O(n)` | worst: `"a…ab…b"` cần `n/2` lượt |
| 3. Ngăn xếp 2 lượt | `O(n)` | `O(n)` | dễ hiểu, tổng quát |
| 4. Đếm thuần | `O(n)` | `O(1)` | **nộp bài** |

Điểm tối đa ≈ `(n/2)·10⁴ = 5·10⁸` — vừa `int` 32 bit, Python không cần lo.

In [17]:
rng = random.Random(1717)
big = "".join(rng.choice("ab" * 6 + "c") for _ in range(10**5))
bad = "a" * 50000 + "b" * 50000
for name, f in [("stack", gain_stack), ("count", gain_count), ("Solution", Solution().maximumGain)]:
    for label, s in [("ngẫu nhiên", big), ("a..ab..b", bad)]:
        t = time.perf_counter()
        r = f(s, 7, 3)
        print(f"{name:>8} | {label:<10}: {time.perf_counter() - t:.3f}s ans={r}")

   stack | ngẫu nhiên: 0.005s ans=232989
   stack | a..ab..b  : 0.003s ans=350000
   count | ngẫu nhiên: 0.003s ans=232989
   count | a..ab..b  : 0.003s ans=350000
Solution | ngẫu nhiên: 0.003s ans=232989
Solution | a..ab..b  : 0.003s ans=350000


## 12. Bộ test

### 12.1 Ví dụ đề + edge case

- `"a"`, `"z"`: không xóa được ⇒ `0`.
- `"ab"` với `y` rất lớn: chỉ có `"ab"` ⇒ `x`.
- `"aba"`, `x=1, y=10`: xóa `"ba"` ⇒ `10` (nếu tham lam sai xóa `"ab"` thì chỉ `1`).
- `x == y`: thứ tự không quan trọng ⇒ `x·min(A, C)` mỗi khúc.
- `"acb"`: chữ `c` chặn ⇒ `0`.
- `"bbbaaa"`, `x=10, y=1`: không có `"ab"` ⇒ `3·1 = 3`.

In [18]:
impls = {
    "brute": gain_brute,
    "replace": gain_replace,
    "stack": gain_stack,
    "count": gain_count,
    "Solution": Solution().maximumGain,
    "SolutionStack": SolutionStack().maximumGain,
}
edge = [
    (("a", 1, 1), 0), (("z", 3, 4), 0), (("ab", 1, 100), 1), (("aba", 1, 10), 10),
    (("abab", 3, 3), 6), (("acb", 5, 5), 0), (("bbbaaa", 10, 1), 3), (("ba", 7, 2), 2),
]
for inp, want in examples + edge:
    for name, f in impls.items():
        assert f(*inp) == want, (name, inp, f(*inp), want)
print("Tất cả cách qua", len(examples) + len(edge), "test cố định")

Tất cả cách qua 10 test cố định


### 12.2 Ngẫu nhiên có seed, so sánh với vét cạn

Chuỗi độ dài `1..12` trên bảng chữ `{a, b}` (nhiều cặp) hoặc `{a, b, c}` (có bức tường), `x, y ∈ [1, 10⁴]` (đúng ràng buộc) và cả `x = y`.

In [19]:
rng = random.Random(7)
cnt = 0
for _ in range(500):
    n = rng.randint(1, 12)
    alpha = rng.choice(["ab", "ab", "abc", "aabbc"])
    s = "".join(rng.choice(alpha) for _ in range(n))
    x = rng.randint(1, 10**4)
    y = x if rng.random() < 0.15 else rng.randint(1, 10**4)
    ref = gain_brute(s, x, y)
    for name, f in impls.items():
        assert f(s, x, y) == ref, (name, s, x, y)
    cnt += 1
for _ in range(100):
    s = "".join(rng.choice("abbaz") for _ in range(rng.randint(500, 3000)))
    x, y = rng.randint(1, 10**4), rng.randint(1, 10**4)
    ref = gain_stack(s, x, y)
    assert gain_count(s, x, y) == ref == gain_replace(s, x, y) == Solution().maximumGain(s, x, y)
    cnt += 1
print("OK", cnt, "bộ ngẫu nhiên khớp nhau")

OK 600 bộ ngẫu nhiên khớp nhau


### 12.3 Kiểm tra tính chất "tổng số lần xóa = Σ min(A, C) mỗi khúc"

In [20]:
def total_removals(s):
    return sum(min(b.count("a"), b.count("b")) for b in re.split(r"[^ab]+", s))

rng = random.Random(8)
for _ in range(300):
    s = "".join(rng.choice("abc") for _ in range(rng.randint(1, 40)))
    # với x = y = 1, điểm chính là số lần xóa
    assert Solution().maximumGain(s, 1, 1) == total_removals(s)
print("Tính chất số lần xóa cố định: đúng")

Tính chất số lần xóa cố định: đúng


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Luôn xóa `"ab"` trước, bất kể `x` và `y`

In [21]:
def wrong_fixed_order(s, x, y):
    rest, c1 = remove_all(s, "a", "b")      # SAI: không so sánh x, y
    _, c2 = remove_all(rest, "b", "a")
    return c1 * x + c2 * y

inp = ("cdbcbbaaabab", 4, 5)
print("sai :", wrong_fixed_order(*inp), "| đúng:", Solution().maximumGain(*inp))

sai : 18 | đúng: 19


**Vì sao sai:** khi `y > x`, mẫu `"ba"` đắt hơn; xóa `"ab"` trước làm "tiêu" mất các chữ có thể tạo `"ba"`. Ví dụ 1 ra `18` thay vì `19`. **Sửa:** chuẩn hóa bằng hoán đổi nếu `x < y`.

### Lỗi 2 — Tham lam "gặp cặp nào xóa cặp đó" từ trái sang phải

In [22]:
def wrong_leftmost(s, x, y):
    st = []; ans = 0
    for c in s:
        if st and st[-1] + c == "ab":
            st.pop(); ans += x
        elif st and st[-1] + c == "ba":
            st.pop(); ans += y
        else:
            st.append(c)
    return ans

inp = ("aba", 1, 10)
print("sai :", wrong_leftmost(*inp), "| đúng:", Solution().maximumGain(*inp))

sai : 1 | đúng: 10


**Vì sao sai:** gặp `"ab"` ở đầu, bản sai xóa ngay (+1) và không còn gì cho `"ba"` (+10). Không phải cứ xóa sớm là tốt — phải **ưu tiên theo giá trị**. **Sửa:** hai lượt (đắt trước, rẻ sau).

### Lỗi 3 — Dùng `replace` một lần

In [23]:
def wrong_replace_once(s, x, y):
    P, Q = ("ab", "ba") if x >= y else ("ba", "ab")
    hi, lo = max(x, y), min(x, y)
    c1 = s.count(P); s = s.replace(P, "")      # SAI: chỉ một lượt
    c2 = s.count(Q); s = s.replace(Q, "")
    return c1 * hi + c2 * lo

inp = ("aabb", 5, 1)
print("sai :", wrong_replace_once(*inp), "| đúng:", Solution().maximumGain(*inp))

sai : 5 | đúng: 10


**Vì sao sai:** `"aabb".replace("ab", "")` cho `"ab"` — cặp mới hình thành sau khi xóa không được xóa tiếp. **Sửa:** lặp đến khi không đổi (chậm), hoặc dùng ngăn xếp / đếm (nhanh).

### Lỗi 4 — Đếm nhưng quên reset ở ký tự khác `a`/`b`

In [24]:
def wrong_no_reset(s, x, y):
    first, second = "a", "b"
    if x < y:
        first, second, x, y = second, first, y, x
    ans = cf = cs = 0
    for c in s:
        if c == first:
            cf += 1
        elif c == second:
            if cf:
                ans += x; cf -= 1
            else:
                cs += 1
        # SAI: bỏ qua ký tự khác mà không chốt khúc / reset
    return ans + y * min(cf, cs)

inp = ("acb", 5, 5)
print("sai :", wrong_no_reset(*inp), "| đúng:", Solution().maximumGain(*inp))

sai : 5 | đúng: 0


**Vì sao sai:** chữ `c` không bao giờ bị xóa nên `a` và `b` ở hai bên **không thể** kề nhau. Bản sai vẫn ghép chúng. **Sửa:** khi gặp ký tự khác, cộng `y·min(cf, cs)` rồi đặt `cf = cs = 0`.

### Lỗi 5 — Lượt 2 chạy trên chuỗi **gốc** thay vì phần còn lại

In [25]:
def wrong_second_on_original(s, x, y):
    first, second = "a", "b"
    if x < y:
        first, second, x, y = second, first, y, x
    _, c1 = remove_all(s, first, second)
    _, c2 = remove_all(s, second, first)      # SAI: phải là phần còn lại
    return c1 * x + c2 * y

inp = ("abab", 3, 2)
print("sai :", wrong_second_on_original(*inp), "| đúng:", Solution().maximumGain(*inp))

sai : 8 | đúng: 6


**Vì sao sai:** các chữ đã bị xóa ở lượt 1 bị dùng lại ở lượt 2 — một chữ `a` bị tính trong hai lần xóa. `"abab"` có 4 chữ nên tối đa 2 lần xóa (6 điểm), bản sai ra `8`. **Sửa:** lượt 2 nhận `rest` từ lượt 1.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Xóa chuỗi con, phần còn lại nối lại, lặp tùy ý"** ⇒ nghĩ ngay tới **ngăn xếp** (LC 1047, 1209, 2696).
- **Hai thao tác với giá trị khác nhau, cùng "tiêu" tài nguyên** ⇒ thử tham lam "giá trị cao trước" rồi chứng minh bằng **tổng số thao tác cố định**.
- **Ký tự không tham gia** ⇒ chia chuỗi thành khúc độc lập.
- **Ngăn xếp chỉ chứa 2 loại ký tự theo thứ tự cố định** ⇒ thay bằng **biến đếm** để có `O(1)` bộ nhớ.
- Chuẩn hóa bằng hoán đổi (`if x < y: swap`) để chỉ viết một nhánh code.
- Kiểm chứng tham lam bằng vét cạn có `lru_cache` trên chuỗi ngắn — rất nhanh để viết.

## 15. Biến thể của bài

1. **LC 1047 — Remove All Adjacent Duplicates In String**: ngăn xếp xóa cặp giống nhau.
2. **LC 2696 — Minimum String Length After Removing Substrings**: xóa `"AB"`/`"CD"`, hỏi độ dài nhỏ nhất (không có điểm) ⇒ một lượt ngăn xếp.
3. **LC 1209 — Remove All Adjacent Duplicates in String II**: xóa `k` ký tự giống nhau liên tiếp ⇒ ngăn xếp `(ký tự, đếm)`.
4. **Hai chữ cái bất kỳ** (`p`, `q` thay cho `a`, `b`) — bài tập 5.
5. **Mẫu dài hơn** (ví dụ `"abc"`) ⇒ ngăn xếp so khớp `k` phần tử cuối; tham lam có thể **không** còn đúng khi các mẫu có độ dài khác nhau.
6. **Chỉ hỏi số lần xóa tối đa** ⇒ `Σ min(A, C)` mỗi khúc, không cần tham lam.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn có nhớ | thử mọi thứ tự xóa | hàm mũ | hàm mũ | đáp án chuẩn cho test nhỏ |
| 2. `replace` lặp | tham lam, xóa bằng thư viện | `O(n²)` | `O(n)` | nhanh để viết, dữ liệu nhỏ |
| 3. Ngăn xếp 2 lượt | tham lam, mô phỏng đúng việc xóa | `O(n)` | `O(n)` | dễ giải thích, mở rộng cho mẫu khác |
| 4. Đếm thuần | ngăn xếp nén thành 2 biến | `O(n)` | `O(1)` | **nộp bài** |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** (a) `("ab", 1, 10)`; (b) `("aba", 1, 10)`; (c) `("abab", 3, 2)`; (d) `("bbaacbab", 2, 3)`.

**Bài 2 (tính tay):** Với `s = "baabab"`, `x = 2`, `y = 5`: xác định mẫu đắt, chạy lượt 1 bằng ngăn xếp (ghi trạng thái), lượt 2, rồi tính điểm.

**Bài 3 (điền TODO):** Hoàn thiện `remove_all_todo(t, p, q)` trả `(phần còn lại, số lần xóa pq)` bằng ngăn xếp.

**Bài 4 (tự viết lại):** Viết lại Cách 4 bằng cách **tách khúc** bằng `re.split` rồi xử lý từng khúc bằng hàm `score_block(block, x, y)`.

**Bài 5 (mini challenge):** Tổng quát hóa: `max_gain_general(s, p, q, x, y)` với hai chữ cái khác nhau `p`, `q`: xóa `p+q` được `x`, xóa `q+p` được `y`. Kiểm tra bằng vét cạn.

In [26]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def remove_all_todo(t, p, q):
    st = []
    cnt = 0
    for c in t:
        # TODO: nếu c == q và đỉnh là p thì pop + đếm, ngược lại append
        pass
    return "".join(st), cnt

In [27]:
# Bài 4 — TODO (chỉ định nghĩa, không gọi)
def score_block_todo(block, x, y):
    # TODO: block chỉ gồm 'a','b'; trả điểm tối đa của riêng khúc này
    pass

def gain_by_blocks_todo(s, x, y):
    # TODO: tổng score_block trên các khúc của re.split(r"[^ab]+", s)
    pass

In [28]:
# Bài 5 — TODO (chỉ định nghĩa, không gọi)
def max_gain_general_todo(s, p, q, x, y):
    # TODO: như Cách 4 nhưng thay 'a','b' bằng p, q
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: xác định mẫu đắt. Mức 2: đếm `min(A, C)` mỗi khúc là số lần xóa. Mức 3: (d) có chữ `c` chia đôi: `"bbaa"` và `"bab"`.

**Bài 2.** Mức 1: `y = 5 > x = 2` ⇒ `"ba"` đắt. Mức 2: ngăn xếp lượt 1 ghép `b` (đỉnh) với `a` (đến). Mức 3: `A = 3, C = 3` ⇒ tổng 3 lần xóa.

**Bài 3.** Mức 1: điều kiện `c == q and st and st[-1] == p`. Mức 2: nhớ kiểm tra `st` không rỗng trước `st[-1]`. Mức 3: trả `"".join(st)`.

**Bài 4.** Mức 1: trong một khúc, dùng `cf`, `cs` như Cách 4 nhưng không cần nhánh "ký tự khác". Mức 2: cuối khúc cộng `y·min(cf, cs)`. Mức 3: nhớ hoán đổi `x, y` và vai trò chữ cái trong `score_block`.

**Bài 5.** Mức 1: thay `first, second = p, q`. Mức 2: brute force tổng quát nhận `p`, `q`. Mức 3: thử với `p = 'x'`, `q = 'y'` trên chuỗi có lẫn `a`, `b`.

## 19. Đáp án tham khảo cho bài tập

**Bài 1:** (a) `1` — chỉ có `"ab"`. (b) `10` — xóa `"ba"`. (c) `6` — xóa `"ab"` hai lần. (d) `x=2 < y=3` ⇒ `"ba"` đắt. Khúc `"bbaa"`: ngăn xếp ghép `ba` 2 lần ⇒ `6`. Khúc `"bab"`: `ba` 1 lần ⇒ `3`, còn `"b"`. Tổng `9`.

In [29]:
assert Solution().maximumGain("ab", 1, 10) == 1
assert Solution().maximumGain("aba", 1, 10) == 10
assert Solution().maximumGain("abab", 3, 2) == 6
assert Solution().maximumGain("bbaacbab", 2, 3) == 9 == gain_brute("bbaacbab", 2, 3)
print("Bài 1 đúng")

Bài 1 đúng


**Bài 2:** `"ba"` đắt (5). Lượt 1 trên `"baabab"`: `b` → `[b]`; `a` ghép `ba` (+1) → `[]`; `a` → `[a]`; `b` → `[a,b]`; `a` ghép `ba` (+1) → `[a]`; `b` → `[a,b]`. Phần còn lại `"ab"`, lượt 2 xóa `"ab"` một lần. Điểm `2·5 + 1·2 = 12`.

In [30]:
rest, c1 = trace_stack("baabab", "b", "a")
rest2, c2 = trace_stack(rest, "a", "b")
assert c1 * 5 + c2 * 2 == 12 == gain_brute("baabab", 2, 5)
print("Bài 2 đúng: 12")

i= 0 c=b: đẩy vào, stack='b'
i= 1 c=a: đỉnh là 'b' -> xóa 'ba', stack='', cnt=1
i= 2 c=a: đẩy vào, stack='a'
i= 3 c=b: đẩy vào, stack='ab'
i= 4 c=a: đỉnh là 'b' -> xóa 'ba', stack='a', cnt=2
i= 5 c=b: đẩy vào, stack='ab'
i= 0 c=a: đẩy vào, stack='a'
i= 1 c=b: đỉnh là 'a' -> xóa 'ab', stack='', cnt=1
Bài 2 đúng: 12


In [31]:
# Bài 3 — đáp án
def remove_all_answer(t, p, q):
    st = []
    cnt = 0
    for c in t:
        if c == q and st and st[-1] == p:
            st.pop()
            cnt += 1
        else:
            st.append(c)
    return "".join(st), cnt

assert remove_all_answer("aabb", "a", "b") == ("", 2)
assert remove_all_answer("cdbcbbaaabab", "b", "a") == ("cdbcab", 3)
print("Bài 3 đúng")

Bài 3 đúng


In [32]:
# Bài 4 — đáp án
def score_block(block, x, y):
    first, second = "a", "b"
    if x < y:
        first, second, x, y = second, first, y, x
    ans = cf = cs = 0
    for c in block:
        if c == first:
            cf += 1
        elif cf:
            ans += x; cf -= 1
        else:
            cs += 1
    return ans + y * min(cf, cs)

def gain_by_blocks(s, x, y):
    return sum(score_block(b, x, y) for b in re.split(r"[^ab]+", s) if b)

rng = random.Random(4)
for _ in range(300):
    s = "".join(rng.choice("abbac") for _ in range(rng.randint(1, 30)))
    x, y = rng.randint(1, 20), rng.randint(1, 20)
    assert gain_by_blocks(s, x, y) == Solution().maximumGain(s, x, y)
print("Bài 4 đúng")

Bài 4 đúng


In [33]:
# Bài 5 — đáp án
def max_gain_general(s, p, q, x, y):
    first, second = p, q
    if x < y:
        first, second, x, y = second, first, y, x
    ans = cf = cs = 0
    for c in s:
        if c == first:
            cf += 1
        elif c == second:
            if cf:
                ans += x; cf -= 1
            else:
                cs += 1
        else:
            ans += y * min(cf, cs); cf = cs = 0
    return ans + y * min(cf, cs)

def brute_general(s, p, q, x, y):
    @lru_cache(maxsize=None)
    def best(t):
        res = 0
        for k in range(len(t) - 1):
            if t[k:k + 2] == p + q:
                res = max(res, x + best(t[:k] + t[k + 2:]))
            elif t[k:k + 2] == q + p:
                res = max(res, y + best(t[:k] + t[k + 2:]))
        return res
    return best(s)

rng = random.Random(5)
for _ in range(300):
    s = "".join(rng.choice("xyyxa") for _ in range(rng.randint(1, 11)))
    x, y = rng.randint(1, 9), rng.randint(1, 9)
    assert max_gain_general(s, "x", "y", x, y) == brute_general(s, "x", "y", x, y)
assert max_gain_general("cdbcbbaaabab", "a", "b", 4, 5) == 19
print("Bài 5 đúng")

Bài 5 đúng


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
# Tham lam: xóa mẫu đắt trước (đếm thuần, O(1) bộ nhớ)
first, second = "a", "b"
if x < y:
    first, second, x, y = second, first, y, x
ans = cf = cs = 0
for c in s:
    if c == first:   cf += 1
    elif c == second:
        if cf: ans += x; cf -= 1
        else:  cs += 1
    else:            ans += y * min(cf, cs); cf = cs = 0
ans += y * min(cf, cs)

# Ngăn xếp xóa mẫu pq liên tục
st = []
for c in t:
    if c == q and st and st[-1] == p: st.pop()
    else: st.append(c)
```

**Checklist trước khi submit:**
- [ ] Đã hoán đổi để xử lý mẫu **điểm cao hơn** trước.
- [ ] Cặp mới hình thành sau khi xóa được xử lý (ngăn xếp/đếm, không phải `replace` một lần).
- [ ] Ký tự khác `a`/`b` **chốt khúc** và reset bộ đếm.
- [ ] Lượt 2 chạy trên **phần còn lại**, không phải chuỗi gốc.
- [ ] Nhớ cộng phần `y·min(cf, cs)` của khúc **cuối cùng**.
- [ ] Thử `"aba"` với `y > x`, `"aabb"`, `"acb"` và 2 ví dụ của đề.